## 简介
上下文编辑中间件，该中间件提供了上下文管理的一种方式。

通过更改发送给模型的消息列表来控制成本。

注意：不会更改消息列表。因此我们只能通过token用量来推测是否对消息列表进行了裁剪。

## 实验组-启用上下文编辑中间件

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit
from langchain.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from dotenv import load_dotenv
load_dotenv()

count = 0

@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    return (f"当前是第 {count} 次调用工具，{city}今天天气晴朗"
            f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
            f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
            f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
            f"真不错，天气非常好，适合出行，这里token挺多的"
            f"可以出门玩，逛街跑步，钓鱼，爬山，一切都很好哈哈哈哈")


agent = create_agent(
    model="deepseek-v4-flash",
    tools=[get_weather],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=50,
                    keep=0,
                ),
            ],
        ),
    ],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}

for i in range(3):
    print("=" * 30, f"当前是第 {i + 1} 轮调用", "=" * 30)
    count = i + 1
    response = agent.invoke({
        "messages": [HumanMessage(f"第 {i + 1} 次询问：今天北京天气如何，一句话回答")]
    }, config=config)

    print("---- 本次返回的 messages ----")
    for msg in response["messages"]:
        # msg.pretty_print()
        if isinstance(msg, AIMessage):
            if not msg.tool_calls:
                print(f"本次token用量: {msg.usage_metadata}")


============================== 当前是第 1 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量: {'input_tokens': 434, 'output_tokens': 58, 'total_tokens': 492, 'input_token_details': {'cache_read': 384}, 'output_token_details': {'reasoning': 46}}
============================== 当前是第 2 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量: {'input_tokens': 434, 'output_tokens': 58, 'total_tokens': 492, 'input_token_details': {'cache_read': 384}, 'output_token_details': {'reasoning': 46}}
本次token用量: {'input_tokens': 754, 'output_tokens': 47, 'total_tokens': 801, 'input_token_details': {'cache_read': 640}, 'output_token_details': {'reasoning': 38}}
============================== 当前是第 3 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量: {'input_tokens': 434, 'output_tokens': 58, 'total_tokens': 492, 'input_token_details': {'cache_read': 384}, 'output_token_details': {'reasoning': 46}}
本次token用量: {'input_tokens': 754, 'output_tokens': 47, 'tot

In [4]:
# 观察 ContextEditingMiddleware 真正发送给模型的临时消息。
# Middleware 列表从外到内执行：ContextEditingMiddleware 先复制并编辑消息，
# 再把编辑后的 request 交给 observe_after_edit，因此这里能看到 "[cleared]"。
from langchain.agents.middleware import wrap_model_call
from langchain.messages import ToolMessage


@wrap_model_call
def observe_after_edit(request, handler):
    tool_messages = [
        message for message in request.messages if isinstance(message, ToolMessage)
    ]

    print(f"\n[即将发送给模型的 ToolMessage 数量: {len(tool_messages)}]")
    if not tool_messages:
        print("  （工具调用前的模型请求，此时还没有工具结果）")

    for index, message in enumerate(tool_messages, 1):
        context_editing = message.response_metadata.get("context_editing", {})
        was_cleared = context_editing.get("cleared", False)
        preview = str(message.content).replace("\n", " ")[:100]
        print(
            f"  {index}. tool={message.name}, "
            f"cleared={was_cleared}, content={preview!r}"
        )

    return handler(request)


# 使用 keep=1：保留最新工具结果，清理更早的结果。
# 这样从第 2 轮的“工具调用后”开始，可以同时观察到：旧结果为 [cleared]，新结果仍保留。
observed_agent = create_agent(
    model="deepseek-v4-flash",
    tools=[get_weather],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=50,
                    keep=1,
                    placeholder="[cleared]",
                )
            ]
        ),
        observe_after_edit,
    ],
    checkpointer=InMemorySaver(),
)

observe_config = {"configurable": {"thread_id": "context-edit-observer"}}

for i in range(3):
    print("\n" + "=" * 24 + f" 观察第 {i + 1} 轮 " + "=" * 24)
    count = i + 1
    observed_agent.invoke(
        {
            "messages": [
                HumanMessage(f"第 {i + 1} 次询问：今天北京天气如何，一句话回答")
            ]
        },
        config=observe_config,
    )

print("\n观察结论：response['messages'] 中仍保存完整工具结果；"
      "只有本单元打印的临时 request.messages 会显示 [cleared]。")



======================== 观察第 1 轮 ========================

[即将发送给模型的 ToolMessage 数量: 0]
  （工具调用前的模型请求，此时还没有工具结果）

[即将发送给模型的 ToolMessage 数量: 1]
  1. tool=get_weather, cleared=False, content='当前是第 1 次调用工具，北京今天天气晴朗天气非常好，北风，非常适合出行，盼望着，盼望着，春天来了。我喜欢春天，你喜欢吗，天气真的很不错万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的真不错，'

======================== 观察第 2 轮 ========================

[即将发送给模型的 ToolMessage 数量: 1]
  1. tool=get_weather, cleared=False, content='当前是第 1 次调用工具，北京今天天气晴朗天气非常好，北风，非常适合出行，盼望着，盼望着，春天来了。我喜欢春天，你喜欢吗，天气真的很不错万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的真不错，'

[即将发送给模型的 ToolMessage 数量: 2]
  1. tool=get_weather, cleared=True, content='[cleared]'
  2. tool=get_weather, cleared=False, content='当前是第 2 次调用工具，北京今天天气晴朗天气非常好，北风，非常适合出行，盼望着，盼望着，春天来了。我喜欢春天，你喜欢吗，天气真的很不错万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的真不错，'

======================== 观察第 3 轮 ========================

[即将发送给模型的 ToolMessage 数量: 2]
  1. tool=get_weather, cleared=True, content='[cleared]'
  2. tool=get_weather, cleared=False, content='当前是第 2 次调用工具，北京今天天气晴朗天气非常好，北风，非常适合出行，盼望着

## 对照组-不启用上下文编辑中间件

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver

# 全局计数器，用于在工具内部追踪这是第几次被触发
count = 0

@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    # 故意返回一段非常冗长、包含大量 Token 的文本，用于测试中间件的 Token 清理/截断功能
    return (f"当前是第 {count} 次调用工具，{city}今天天气晴朗"
            f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
            f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
            f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
            f"真不错，天气非常好，适合出行，这里token挺多的"
            f"可以出门玩，逛街跑步，钓鱼，爬山，一切都很好哈哈哈哈")


agent = create_agent(
    model="deepseek-v4-flash",
    tools=[get_weather],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}

for i in range(3):
    print("=" * 30, f"当前是第 {i + 1} 轮调用", "=" * 30)
    count = i + 1
    response = agent.invoke({
        "messages": [HumanMessage(f"第 {i + 1} 次询问：今天北京天气如何，一句话回答")]
    }, config=config)

    print("---- 本次返回的 messages ----")
    for msg in response["messages"]:
        # msg.pretty_print()
        if isinstance(msg, AIMessage):
            if not msg.tool_calls:
                print(f"本次token用量: {msg.usage_metadata}")


============================== 当前是第 1 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量: {'input_tokens': 455, 'output_tokens': 42, 'total_tokens': 497, 'input_token_details': {'cache_read': 256}, 'output_token_details': {'reasoning': 25}}
============================== 当前是第 2 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量: {'input_tokens': 455, 'output_tokens': 42, 'total_tokens': 497, 'input_token_details': {'cache_read': 256}, 'output_token_details': {'reasoning': 25}}
本次token用量: {'input_tokens': 626, 'output_tokens': 24, 'total_tokens': 650, 'input_token_details': {'cache_read': 384}, 'output_token_details': {'reasoning': 9}}
============================== 当前是第 3 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量: {'input_tokens': 455, 'output_tokens': 42, 'total_tokens': 497, 'input_token_details': {'cache_read': 256}, 'output_token_details': {'reasoning': 25}}
本次token用量: {'input_tokens': 626, 'output_tokens': 24, 'tota